In [12]:
from torch.utils.data import Dataset,DataLoader
from torch import nn
import os
from PIL import Image
import torch
import matplotlib.pyplot as plt
from torchvision import models
from random import randint
from tqdm import tqdm
import yaml
from torchvision import transforms

In [13]:
option_path=fr'/home/artemybombastic/MyGit/KD_Summer_Work/config.yml'
device='cuda' if torch.cuda.is_available() else 'cpu'
device='cpu'
with open(option_path,'r') as file_option:
    option=yaml.safe_load(file_option)


In [4]:
class Boot_Rotate_Dataset(Dataset):
  def __init__(self,path):
    super(Boot_Rotate_Dataset,self).__init__()

    dirs=[os.path.join(path,dir) for dir in os.listdir(path)]

    self.all_images=[]
    for dir in dirs:
      images=[os.path.join(dir,img) for img in os.listdir(dir) if 'done' in img]
      self.all_images+=images

    self.trans=transforms.Compose([

        transforms.Resize((762,1100))
    ])
    self.tensor_trans=transforms.Compose([

        transforms.Resize((762,1100)),
        transforms.ToTensor()
    ])


  def __len__(self):
    return len(self.all_images)

  def __getitem__(self,idx):

    img=Image.open(self.all_images[idx])

    if randint(0,1):
      degr=randint(0,10)
      img=img.rotate(degr,expand=True)
      #img=self.trans(img)
      new_width,new_height=(610,932)

    else:
      degr=randint(350,360)
      img=img.rotate(degr,expand=True)
      #img=self.trans(img)
      new_width,new_height=(610,932)

    width,height=img.size
    left=(width-new_width)//2
    top=(height-new_height)//2
    right=left+new_width
    bottom=top+new_height
    img=img.crop((left,top,right,bottom))
    tensor_img=self.tensor_trans(img)
    return {
        "img": tensor_img,
        "label":torch.FloatTensor([degr])
        }

In [5]:
def Train_degr_model(model,dataloader,loss_func,optimizer,device):
    #loss_item=0#костыль
    model=model.to(device)
    sigm=nn.Sigmoid()
    for batch in (pbar:=tqdm(dataloader)):
        optimizer.zero_grad()
        pred=sigm(model(batch['img'].to(device)))
        #print(pred,batch['label'])


        loss=loss_func(pred,batch['label'].to(device))
        loss_item=loss.item()
        loss.backward()
        optimizer.step()
        pbar.set_description(f'loss: {loss_item}')
        try:
            torch.save(model.state_dict(),f'/home/artemybombastic/MyGit/KD_Data/TransformData//degr_net_{device}.pth')
        except:
            print('ошибка сохранения весов')

In [6]:
class Degree_Net(nn.Module):
    def __init__(self,input_size,hidden_size):
        super(Degree_Net,self).__init__()
    
        self.lay0=nn.Sequential(
            nn.Conv2d(input_size,hidden_size,3,padding=1),
            nn.BatchNorm2d(hidden_size),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2)
        )
        
        self.lay1=nn.Sequential(
            nn.Conv2d(hidden_size,hidden_size*2,3,padding=1),
        
            nn.BatchNorm2d(hidden_size*2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2)
        )
        
        self.lay2=nn.Sequential(
            nn.Conv2d(hidden_size*2,hidden_size*4,3,padding=1),
            nn.BatchNorm2d(hidden_size*4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2)
        )
        
        self.lay3=nn.Sequential(
            nn.Conv2d(hidden_size*4,hidden_size*8,3,padding=1),
            nn.BatchNorm2d(hidden_size*8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2)
        )
        
        self.linear_lay=nn.Sequential(
            nn.Flatten(),
            nn.Linear(3214080,128),
            nn.ReLU(),
            nn.Linear(128,1)
        )

    def forward(self,x):
        print(x.shape)
        x0=self.lay0(x)
        print(x0.shape)
    
        x1=self.lay1(x0)
          
        print(x1.shape)
    
        x2=self.lay2(x1)
        print(x2.shape)
    
        x3=self.lay3(x2)
        print(x3.shape)
        
    
        final_x=self.linear_lay(x2)
        print(final_x.shape)
    
        return final_x

In [9]:
class Transfered_Resnet50(nn.Module):
    def __init__(self):
        super().__init__()
        self.core=models.resnet50(pretrained=True)

        lin_shape=self.core.fc.in_features
        self.core.fc=nn.Sequential(
            nn.Linear(lin_shape,512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512,1)
            
        )
            
    def forward(self,x):
        x=self.core(x)
        return x

        


    

In [18]:
degr_dataset=Boot_Rotate_Dataset(option['Trans']['data_path'])
degr_dataloader=DataLoader(degr_dataset,batch_size=4,shuffle=True,drop_last=True)

In [7]:
#model=Degree_Net(3,64)
resnet_model=Transfered_Resnet50()#loss:93636.875,30000,63915.5
loss_func=nn.MSELoss()
optimizer=torch.optim.AdamW(resnet_model.parameters())
device='cpu'

/usr/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
Train_degr_model(model=resnet_model,dataloader=degr_dataloader,loss_func=loss_func,optimizer=optimizer,device=device)

loss: 39.0:  99%|████████████████████████████████████▊| 321/323 [1:27:31<00:21, 10.64s/it]

In [12]:
bt=0
for i in degr_dataloader:
    bt=i
    break

In [30]:
plt.imshow(bt['img'][4].permute(1,2,0))

IndexError: index 4 is out of bounds for dimension 0 with size 4